# Bank Churn — TPOT AutoML Search

TPOT 0.12.2 · population 40 · 15 generations · mutation 0.9 / crossover 0.1 · ROC-AUC · same 5 CV folds as Optuna · no early stop or time limit. The holdout set is not used during the search.

In [1]:
import sys, re, time
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Datasets").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import warnings
warnings.filterwarnings("ignore")
import joblib
import numpy as np
import pandas as pd
import sklearn, tpot
from sklearn.base import clone
from sklearn.model_selection import cross_validate
from tpot import TPOTClassifier

from churn_utils import (RANDOM_STATE, RESULTS_DIR, PROJECT_ROOT, load_data, split_data,
                         make_tpot_encoder, tuning_cv, classification_metrics, save_json)

print("TPOT", tpot.__version__, "| scikit-learn", sklearn.__version__, "| Python", sys.version.split()[0])
np.random.seed(RANDOM_STATE)

TPOT 0.12.2 | scikit-learn 1.9.1 | Python 3.11.16


## 1. Data split and encoding

In [2]:
X, y = load_data()
X_train, X_test, y_train, y_test = split_data(X, y)

encoder = make_tpot_encoder().fit(X_train)
feature_names = list(encoder.get_feature_names_out())
Xtr = encoder.transform(X_train)
Xte = encoder.transform(X_test)
print("Features:", feature_names)
print(f"X_train {Xtr.shape} (churn {y_train.mean()*100:.2f}%) | X_test {Xte.shape} (churn {y_test.mean()*100:.2f}%)")

Features: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary', 'Geography_Germany', 'Geography_Spain', 'Gender_Male']
X_train (12000, 11) (churn 20.38%) | X_test (3000, 11) (churn 20.40%)


## 2. Evolutionary search

In [3]:
tpot_model = TPOTClassifier(
    generations=15,
    population_size=40,
    mutation_rate=0.90,
    crossover_rate=0.10,
    scoring="roc_auc",
    cv=tuning_cv(),
    early_stop=None,
    max_time_mins=None,
    max_eval_time_mins=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbosity=2,
)
t0 = time.time()
tpot_model.fit(Xtr, y_train.values)
search_minutes = (time.time() - t0) / 60
print(f"\nSearch time: {search_minutes:.1f} min")
print(f"Pipelines evaluated: {len(tpot_model.evaluated_individuals_)}")
print(f"Best pipeline internal 5-fold CV ROC-AUC (TPOT search score): {tpot_model._optimized_pipeline_score:.4f}")
print("\nBest pipeline:")
print(tpot_model.fitted_pipeline_)

Generation 1 - Current best internal CV score: 0.933374455130075
Generation 2 - Current best internal CV score: 0.9336760421739383
Generation 3 - Current best internal CV score: 0.9336760421739383
Generation 4 - Current best internal CV score: 0.9343364417653023
Generation 5 - Current best internal CV score: 0.9344117869772595
Generation 6 - Current best internal CV score: 0.9344117869772595
Generation 7 - Current best internal CV score: 0.9351539537928488
Generation 8 - Current best internal CV score: 0.9352173575466406
Generation 9 - Current best internal CV score: 0.9352173575466406
Generation 10 - Current best internal CV score: 0.9354867299884531
Generation 11 - Current best internal CV score: 0.9356956813054982
Generation 12 - Current best internal CV score: 0.9356956813054982
Generation 13 - Current best internal CV score: 0.9356956813054982
Generation 14 - Current best internal CV score: 0.9361667113005474
Generation 15 - Current best internal CV score: 0.9361667113005474
Best 

## 3. CV stability of the selected pipeline
Optimistic (best of all evaluated candidates); the fair comparison is in `train_test_models.ipynb`.

In [4]:
best_pipeline = clone(tpot_model.fitted_pipeline_)
cv_res = cross_validate(best_pipeline, Xtr, y_train.values, cv=tuning_cv(), scoring="roc_auc", n_jobs=-1)
print(f"5-fold CV ROC-AUC of the selected pipeline: {cv_res['test_score'].mean():.4f} ± {cv_res['test_score'].std():.4f}")
print("Per fold:", np.round(cv_res["test_score"], 4))

5-fold CV ROC-AUC of the selected pipeline: 0.9362 ± 0.0051
Per fold: [0.939  0.9433 0.932  0.929  0.9376]


## 4. Holdout test evaluation

In [5]:
proba = tpot_model.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
holdout = classification_metrics(y_test, proba, pred)
for k, v in holdout.items():
    print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")

ROC-AUC     : 0.9324
PR-AUC      : 0.8173
Accuracy    : 0.8967
Precision   : 0.8044
Recall      : 0.6520
Specificity : 0.9594
F1-Score    : 0.7202
G-Mean      : 0.7909
TN          : 2291
FP          : 97
FN          : 213
TP          : 399


## 5. Export
`Results/tpot_best_pipeline.joblib`, `Results/tpot_search_summary.json` and the standalone `tpot_best_pipeline.py`.

In [6]:
joblib.dump(clone(tpot_model.fitted_pipeline_), RESULTS_DIR / "tpot_best_pipeline.joblib")

save_json({
    "tpot_version": tpot.__version__,
    "config": {"generations": 15, "population_size": 40, "offspring_size": 40, "mutation_rate": 0.9,
               "crossover_rate": 0.1, "scoring": "roc_auc", "cv": "StratifiedKFold(5, shuffle, seed 42)",
               "early_stop": None, "max_time_mins": None, "max_eval_time_mins": 5},
    "search_minutes": search_minutes,
    "pipelines_evaluated": len(tpot_model.evaluated_individuals_),
    "search_cv_roc_auc": tpot_model._optimized_pipeline_score,
    "refit_cv_roc_auc_mean": float(cv_res["test_score"].mean()),
    "refit_cv_roc_auc_std": float(cv_res["test_score"].std()),
    "best_pipeline": str(tpot_model.fitted_pipeline_),
    "holdout": holdout,
}, RESULTS_DIR / "tpot_search_summary.json")

# Build a runnable script from TPOT's exported code (imports + pipeline definition)
code = tpot_model.export()
imports = [l for l in code.splitlines() if l.startswith(("import ", "from "))
           and "train_test_split" not in l]
pipeline_def = code[code.index("exported_pipeline ="):code.index("exported_pipeline.fit")].rstrip()

script = f'''"""Best pipeline found by TPOT (generated by Notebooks/tpot_test.ipynb — do not edit by hand).

Refits the pipeline on all labelled data (Datasets/train_processed.csv) and writes
Submissions/tpot_submission.csv for the Kaggle test set.

Usage:  python tpot_best_pipeline.py
"""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

{chr(10).join(imports)}
from sklearn.pipeline import Pipeline
from churn_utils import load_data, load_kaggle_test, make_tpot_encoder, SUBMISSION_DIR

{pipeline_def}

if __name__ == "__main__":
    X, y = load_data()
    ids, X_kaggle = load_kaggle_test()
    model = Pipeline([("encoder", make_tpot_encoder()), ("tpot", exported_pipeline)])
    model.fit(X, y)
    out = SUBMISSION_DIR / "tpot_submission.csv"
    pd.DataFrame({{"id": ids, "Exited": model.predict_proba(X_kaggle)[:, 1]}}).to_csv(out, index=False)
    print("Saved", out)
'''
(PROJECT_ROOT / "tpot_best_pipeline.py").write_text(script)
print(script)

"""Best pipeline found by TPOT (generated by Notebooks/tpot_test.ipynb — do not edit by hand).

Refits the pipeline on all labelled data (Datasets/train_processed.csv) and writes
Submissions/tpot_submission.csv for the Kaggle test set.

Usage:  python tpot_best_pipeline.py
"""
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.feature_selection import SelectPercentile, f_classif
from sklearn.pipeline import make_pipeline
from tpot.export_utils import set_param_recursive
from sklearn.pipeline import Pipeline
from churn_utils import load_data, load_kaggle_test, make_tpot_encoder, SUBMISSION_DIR

exported_pipeline = make_pipeline(
    SelectPercentile(score_func=f_classif, percentile=53),
    GradientBoostingClassifier(learning_rate=0.1, max_depth=4, max_features=0.35000000000000003, min_samples_leaf=15, min_samples_spl